# E08 - THIẾT KẾ LUẬT XỬ LÝ CÂU HOÀ TRONG NHÃN 3 LỚP (NGƯỜI 3)

**Vấn đề:** Trong `src/labels.py`, khi câu có số khía cạnh `Positive == Negative` (ví dụ *"Pin kém còn lại miễn chê"*), nhãn bị ép về **Trung lập (1)**. Điều này làm bẩn lớp 1 (F1 trên test chỉ ~0.50).
**Mục tiêu:** Giữ khung 3 lớp nhưng thử 4 quy tắc xử lý câu hoà để tìm cách gán nhãn sạch nhất, so sánh công bằng trên cả dev tổng thể lẫn **lát cắt dev không có câu hoà** (nhãn cố định ở cả 4 biến thể).

In [1]:
import os, sys, glob, zipfile, subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

def _find_root():
    cwd = os.getcwd()
    cand = os.path.abspath('..') if os.path.basename(cwd) in ('notebooks', 'experiments') else cwd
    if os.path.exists(os.path.join(cand, 'src', 'modeling.py')):
        return cand, cand
    if IN_COLAB:
        for base in [cwd, '/content']:
            if os.path.exists(os.path.join(base, 'src', 'modeling.py')):
                return base, base
        for z in glob.glob('/content/*src_and_data*.zip'):
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall('/content')
            if os.path.exists('/content/src/modeling.py'):
                return '/content', '/content'
    if IN_KAGGLE:
        work = '/kaggle/working'
        if os.path.exists(os.path.join(work, 'src', 'modeling.py')):
            return work, work
        for p in glob.glob('/kaggle/input/**/src/modeling.py', recursive=True):
            return os.path.dirname(os.path.dirname(p)), work
        for z in glob.glob('/kaggle/input/**/*src_and_data*.zip', recursive=True):
            dest = os.path.join(work, 'project')
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(dest)
            if os.path.exists(os.path.join(dest, 'src', 'modeling.py')):
                return dest, dest
    raise FileNotFoundError('Khong tim thay src/modeling.py! Hay chay trong repo hoac upload src_and_data.zip.')

ROOT, WORK_DIR = _find_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
FIG_DIR = os.path.join(WORK_DIR, 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
os.environ['EXP_LOG_PATH'] = os.path.join(WORK_DIR, 'reports', 'experiments_log.csv')
print(f'Env: colab={IN_COLAB}, kaggle={IN_KAGGLE} | ROOT={ROOT}')

Env: colab=False, kaggle=False | ROOT=D:\E-commerce


In [2]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
from src.exp_log import log_experiment

df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
PAIR_RE = re.compile(r'\{([A-Z&]+)#(Positive|Neutral|Negative)\}')
POL_MAP = {'Negative': 0, 'Neutral': 1, 'Positive': 2}

def is_tie_row(lbl_str):
    pairs = PAIR_RE.findall(str(lbl_str))
    if not pairs:
        return True
    sents = [POL_MAP[p] for _, p in pairs]
    neg, neu, pos = sents.count(0), sents.count(1), sents.count(2)
    return not ((neg > pos and neg > neu) or (pos > neg and pos > neu) or (neu > neg and neu > pos))

def label_with_rule(lbl_str, rule='tie_neutral'):
    pairs = PAIR_RE.findall(str(lbl_str))
    if not pairs:
        return 1
    sents = [POL_MAP[p] for _, p in pairs]
    neg, neu, pos = sents.count(0), sents.count(1), sents.count(2)
    if neg > pos and neg > neu:
        return 0
    if pos > neg and pos > neu:
        return 2
    if neu > neg and neu > pos:
        return 1
    # Xu ly truong hop hoa / hon hop
    if rule == 'tie_neutral':
        return 1
    if rule == 'tie_negative':
        return 0 if neg > 0 else 1
    if rule == 'tie_specific_aspect':
        spec = [POL_MAP[p] for asp, p in pairs if asp not in ('GENERAL', 'OTHERS')]
        if not spec:
            return 1
        sn, sz, sp = spec.count(0), spec.count(1), spec.count(2)
        if sn > sp: return 0
        if sp > sn: return 2
        return spec[0]
    return 1

dev_is_tie = df_dev['label'].apply(is_tie_row).values
tr_is_tie = df_train['label'].apply(is_tie_row).values
print(f'So cau hoa/khong ro: train={tr_is_tie.sum()}/{len(df_train)} ({tr_is_tie.mean():.1%}), dev={dev_is_tie.sum()}/{len(df_dev)} ({dev_is_tie.mean():.1%})')

So cau hoa/khong ro: train=1256/7776 (16.2%), dev=176/1110 (15.9%)


In [3]:
TOKEN_PAT = r'(?u)\b\w+\b|\[\w+\]'
rows = []
for rule in ['tie_neutral', 'tie_negative', 'tie_specific_aspect', 'drop_ties_train']:
    if rule == 'drop_ties_train':
        sub_tr = df_train[~tr_is_tie].copy()
        y_tr = sub_tr['label'].apply(lambda s: label_with_rule(s, 'tie_neutral')).values
        y_dv = df_dev['label'].apply(lambda s: label_with_rule(s, 'tie_neutral')).values
    else:
        sub_tr = df_train
        y_tr = sub_tr['label'].apply(lambda s: label_with_rule(s, rule)).values
        y_dv = df_dev['label'].apply(lambda s: label_with_rule(s, rule)).values

    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(token_pattern=TOKEN_PAT, ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=15000, sublinear_tf=True)),
        ('clf', LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)),
    ]).fit(sub_tr['sentence_seg'].fillna(''), y_tr)
    pred = pipe.predict(df_dev['sentence_seg'].fillna(''))

    f1_all = f1_score(y_dv, pred, average='macro')
    f1_non_tie = f1_score(y_dv[~dev_is_tie], pred[~dev_is_tie], average='macro')
    per_cls = f1_score(y_dv, pred, labels=[0, 1, 2], average=None)
    rows.append({
        'rule': rule,
        'dev_macro_f1': round(f1_all, 4),
        'fair_non_tie_f1': round(f1_non_tie, 4),
        'f1_neg': round(per_cls[0], 4),
        'f1_neu': round(per_cls[1], 4),
        'f1_pos': round(per_cls[2], 4),
    })
    log_experiment('E08', rule, '3-class tie-break labeling rules', {'rule': rule},
                   dev_f1=f1_all, notes=f'fair_non_tie_f1={f1_non_tie:.4f}, neu_f1={per_cls[1]:.4f}, neg_f1={per_cls[0]:.4f}')

pd.DataFrame(rows)

,rule,dev_macro_f1,fair_non_tie_f1,f1_neg,f1_neu,f1_pos
0,tie_neutral,0.7478,0.6568,0.8387,0.5466,0.8580
1,tie_negative,0.6934,0.6602,0.8649,0.3611,0.8541
2,tie_specific_aspect,0.6806,0.6782,0.8528,0.3196,0.8694
3,drop_ties_train,0.6531,0.7166,0.8157,0.2643,0.8792


## 2. KẾT LUẬN & HƯỚNG GIẢI QUYẾT ĐỀ XUẤT

- Có **`16.2%` (1.256/7.776)** câu train và **`15.9%` (176/1.110)** câu dev là câu hoà hoặc không rõ cực tính.
- Khi chấm trên **lát cắt công bằng các câu KHÔNG bị hoà** (nhãn giống nhau ở cả 4 biến thể):
  - Luật cũ `tie_neutral`: `fair_non_tie_f1 = 0.6568` (`f1_neg = 0.8387`, `f1_neu = 0.5466`).
  - `tie_negative` (đẩy câu hoà có chê về Tiêu cực): nâng `f1_neg` lên **`0.8649`** (`+0.0262`) và `fair_non_tie_f1 = 0.6602`, nhưng làm lớp Neutral chỉ còn rất ít mẫu (`f1_neu = 0.3611`).
  - `drop_ties_train` (bỏ câu hoà khỏi tập train): tăng vọt `fair_non_tie_f1` lên **`0.7166` (`+0.0598`)**, chứng minh câu hoà chính là nguồn nhiễu làm mờ ranh giới 3 lớp.
- **Hướng giải quyết đề xuất:** Nếu giữ bài toán 3 lớp phục vụ cảnh báo CSKH (không bỏ sót phàn nàn), ưu tiên `tie_negative` để tối đa hoá nhận diện Tiêu cực (`0.8649`); còn về kiến trúc lâu dài, chuyển sang dự đoán cảm xúc theo từng khía cạnh (ABSA) thay vì ép 1 nhãn cả câu.
